<b>Importing Tools</b>

In [10]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from sklearn.model_selection import train_test_split

<b>Loading Data</b>


In [11]:
# *col names in of the data
col_names = ['prognosis','fatigue', 'vomiting', 'high_fever', 'loss_of_appetite', 'nausea', 'headache', 'abdominal_pain', 'yellowish_skin', 'yellowing_of_eyes', 'chills', 'skin_rash']
# *loading data in a pandas dataframe data struct using col_names as its column names
data = pd.read_csv('data/Sintomas_menor.csv',skiprows=1,header=None,names = col_names)

<b>Node class</b>

In [12]:
@dataclass(init=False)
class Node:
    def __init__(self,left=None,right=None,info_gain=None,threshold=None,feature_index=None,value=None) -> None:
        # *for decision Node
        self.left = left
        self.right = right
        self.info_gain = info_gain
        self.threshold = threshold
        self.feature_index = feature_index

        # *for leaf Node
        self.value = value

<b>Tree class</b>

In [13]:
class DecisionTreeClassifier:
    def __init__(self, min_samples_split=2, max_depth=2) -> None:
        '''tree constructor'''
        # *initialize root node
        self.root = None

        # *stopping conditions
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def buildTree(self, dataset: np.ndarray, curr_depth=0) -> Node:
        '''recursive function to build the tree'''
        X = dataset[:,:-1]
        num_samples, num_features = np.shape(X)
        # *split until conditions are met
        if num_samples >= self.min_samples_split and curr_depth <= self.max_depth:
            # *find the best split
            best_split = self.getBestSplit(dataset, num_samples, num_features)
            # *check if info gain is positive
            if best_split['info_gain'] > 0:
                # *recur left
                left_subtree = self.buildTree(best_split['dataset_left'], curr_depth+1)
                # *recur right
                right_subtree = self.buildTree(best_split['dataset_right'], curr_depth+1)
                # *return decision Node
                return Node(
                    feature_index=best_split['feature_index'],
                    threshold=best_split['threshold'],
                    info_gain=best_split['info_gain'],
                    left=left_subtree,
                    right=right_subtree)

        Y = dataset[:, -1]
        leaf_value = self.calculateLeafValue(Y)
        return Node(value=leaf_value)

    def getBestSplit(self,dataset:np.ndarray,num_samples:int,num_features:int) -> dict:
        '''method that return the best split of the data
        \n Args:
            dataset: the dataset to be used in order to get the best split
            num_samples: a int tha represents the number of samples of the given dateset
            num_features: a float number that represents the number of features of the given dataset
        \n Returns:
            A dict containing the best split in this format:
                'feature_index' : feature_index
                'threshold' : threshold
                'dataset_left' : dataset_left
                'dataset_right' : dataset_right
                'info_gain' : curr_info_gain
        '''
        best_split = {
            'info_gain':0
        }
        max_info_gain = -float('inf')
        
        # *loop all the features
        for feature_index in range(num_features):
            feature_values = dataset[:,feature_index]
            possible_thresholds = np.unique(feature_values)
            # *loop for all feature values present on data
            for threshold in possible_thresholds:
                # *get current split
                dataset_left,dataset_right = self.split(dataset,feature_index,threshold)
                # *check if the childs are not null
                if len(dataset_left)>0 and len(dataset_right)>0:
                    y,left_y,right_y = dataset[:,-1],dataset_left[:,-1],dataset_right[:,-1]
                    # *calculate info gain
                    curr_info_gain = self.informationGain(y,left_y,right_y,'gini')
                    # *update best split if needed
                    if curr_info_gain > max_info_gain:
                        best_split['feature_index'] = feature_index
                        best_split['threshold'] = threshold
                        best_split['dataset_left'] = dataset_left
                        best_split['dataset_right'] = dataset_right
                        best_split['info_gain'] = curr_info_gain
                        max_info_gain = curr_info_gain
        
        return best_split
    
    def split(self,dataset,feature_index,threshold) -> tuple:
        '''method to split the data'''
        dataset_left = np.array([row for row in dataset if row[feature_index] <= threshold])
        dataset_right =  np.array([row for row in dataset if row[feature_index] > threshold])
        return dataset_left,dataset_right
    
    def informationGain(self,parent,l_child,r_child,mode='entropy') -> float:
        '''method that calculates the info gain
            \nargs:
            parent: parent node
            l_child: left child node of parent
            r_child: right child node of parent
            mode: a string that chooses the algorithm of the calculated info gain
                \n- 'gini' => Gini's ppurity index algorithm
                \n- 'entropy' => entropy algorithm
            \nreturns a float representing the info gain of the parent node.
        '''
        weight_l = len(l_child) / len(parent)
        weight_r = len(r_child) / len(parent)
        if mode == 'gini':
            gain = self.giniIndex(parent) - ((self.giniIndex(l_child)*weight_l) + (self.giniIndex(r_child)*weight_r))
        else:
            gain = self.entropy(parent) - ((self.entropy(l_child)*weight_l) + (self.entropy(r_child)*weight_r))
        return gain

    def entropy(self,y:np.ndarray) ->float: 
        '''method that calculates the entropy of an threshold'''
        class_labels = np.unique(y)
        entropy = 0
        for clabel in class_labels:
            p_cls = len(y[y==clabel])/len(y)
            entropy += -p_cls * np.log2(p_cls)
        return entropy
    
    def giniIndex(self,y:np.ndarray) -> float:
        ''' method that calculates the giniIdex of an threshold'''
        class_labels = np.unique(y)
        gini = 0
        for clabel in class_labels:
            p_cls = len(y[y==clabel])/len(y)
            gini += p_cls**2
        return 1 - gini
    
    def calculateLeafValue(self, y: np.ndarray):
        '''method to calculate the leaf value'''
        unique_classes, counts = np.unique(y, return_counts=True) 
        majority_class = unique_classes[np.argmax(counts)]
        return majority_class

    def printTree(self,node:Node=None,indent:str='|') -> None: #? prints the generated Tree
        '''method to print the tree'''
        if not node:
            node = self.root
        
        if node.value is not None:
            print(node.value)
        else:
            print(f'{col_names[node.feature_index+1]}?')
            print(f'{indent}left:',end='')
            self.printTree(node.left,indent+'|')
            print(f'{indent}right:',end='')
            self.printTree(node.right,indent+'|')
    
    def fit(self,X,Y) -> None:
        '''method to train the tree and call the buildTree method'''
        dataset = np.concatenate((X,Y),axis=1)
        self.root = self.buildTree(dataset=dataset)
    
    def predict(self,X):
        '''method to predict a new dataset'''
        preditions = [self.make_prediction(x,self.root) for x in X]
        return preditions
    
    def make_prediction(self,x,node:Node):
        '''method to predict a single datapoint'''
        if node.value != None: return node.value
        feature_val = x[node.feature_index]
        if feature_val<= node.threshold:
            return self.make_prediction(x,node.left)
        else:
            return self.make_prediction(x,node.right)
    
    def ask(self):
        '''method ta asks the user and predicts the feature
        based on the given yes/no answers'''
        self.__askHelper(self.root)
    
    def __askHelper(self,node:Node):
        '''ask helper recursive method'''
        if(node == None): return # ?checks if it is a null node
        if node.feature_index != None: # ?checks if it is a feature node or a leaf node
            ans = input(f'Sympton: {col_names[node.feature_index+1]}? (yes/no)').lower()
            if ans == 'yes':
                self.__askHelper(node.right)
            elif ans == 'no':
                self.__askHelper(node.left)
            else:
                print('Invalid answer. Please answer with yes or no.')
                self.__askHelper(node)
        else:
            print(f'The diagnosis is: {node.value}\n')

<b>Train-Test split</b>

In [14]:
"""
Prepares data for machine learning tasks by separating features and target variable and splitting into training and testing sets.

Parameters:
    data (pandas.DataFrame): Input DataFrame containing the data.
    
Returns:
    X_train (pandas.DataFrame): DataFrame containing the features for the training set.
    X_test (pandas.DataFrame): DataFrame containing the features for the testing set.
    Y_train (numpy.ndarray): Array containing the target variable values for the training set.
    Y_test (numpy.ndarray): Array containing the target variable values for the testing set.
"""
A = data.iloc[:,1:]
B = data.iloc[:,0].values.reshape(-1,1)
X_train,X_test,Y_train,Y_test = train_test_split(A,B,test_size=.2,random_state=21)

<b>Fit the model</b>

In [15]:
# *creating and training the tree
classifier = DecisionTreeClassifier(min_samples_split=2,max_depth=2000) # ?creating tree using its constructor
classifier.fit(X_train,Y_train) # ?training it

<b>Asking user</b>

In [16]:
#asking user
flag = True
while(True):
    print('>>--Sistema--<<')
    if flag:
        op = input('Would you like to make a prognosis? (yes/no)').lower()
        flag = False
    else:
        op = input('Would you like to make another prognosis? (yes/no)').lower()
    match op:
        case 'yes':
            classifier.ask()
        case 'no':
            print('Bye bye!')
            break
        case _:
            print('Invalid answer. Please answer with yes or no.')
            continue
op = input('Do you want to print the tree? (yes/no)').lower()
if op == 'yes':
    classifier.printTree()

>>--Sistema--<<
The diagnosis is: Tuberculosis

>>--Sistema--<<
Bye bye!
skin_rash?
|left:chills?
||left:high_fever?
|||left:fatigue?
||||left:vomiting?
|||||left:abdominal_pain?
||||||left:Hepatitis C
||||||right:nausea?
|||||||left:Hepatitis B
|||||||right:Chronic cholestasis
|||||right:yellowing_of_eyes?
||||||left:Chronic cholestasis
||||||right:abdominal_pain?
|||||||left:hepatitis A
|||||||right:nausea?
||||||||left:hepatitis A
||||||||right:loss_of_appetite?
|||||||||left:hepatitis A
|||||||||right:yellowish_skin?
||||||||||left:Chronic cholestasis
||||||||||right:Chronic cholestasis
||||right:vomiting?
|||||left:abdominal_pain?
||||||left:nausea?
|||||||left:Hepatitis B
|||||||right:Hepatitis C
||||||right:nausea?
|||||||left:Hepatitis B
|||||||right:Hepatitis D
|||||right:headache?
||||||left:loss_of_appetite?
|||||||left:nausea?
||||||||left:Jaundice
||||||||right:Hepatitis D
|||||||right:nausea?
||||||||left:Hepatitis D
||||||||right:yellowing_of_eyes?
|||||||||left:Hepatiti

In [17]:
classifier.printTree()

skin_rash?
|left:chills?
||left:high_fever?
|||left:fatigue?
||||left:vomiting?
|||||left:abdominal_pain?
||||||left:Hepatitis C
||||||right:nausea?
|||||||left:Hepatitis B
|||||||right:Chronic cholestasis
|||||right:yellowing_of_eyes?
||||||left:Chronic cholestasis
||||||right:abdominal_pain?
|||||||left:hepatitis A
|||||||right:nausea?
||||||||left:hepatitis A
||||||||right:loss_of_appetite?
|||||||||left:hepatitis A
|||||||||right:yellowish_skin?
||||||||||left:Chronic cholestasis
||||||||||right:Chronic cholestasis
||||right:vomiting?
|||||left:abdominal_pain?
||||||left:nausea?
|||||||left:Hepatitis B
|||||||right:Hepatitis C
||||||right:nausea?
|||||||left:Hepatitis B
|||||||right:Hepatitis D
|||||right:headache?
||||||left:loss_of_appetite?
|||||||left:nausea?
||||||||left:Jaundice
||||||||right:Hepatitis D
|||||||right:nausea?
||||||||left:Hepatitis D
||||||||right:yellowing_of_eyes?
|||||||||left:Hepatitis D
|||||||||right:abdominal_pain?
||||||||||left:Hepatitis D
||||||||||r